# HelmX Crash Detection — Training

Trains the hybrid crash-detection model: a shared Conv1D encoder with an unsupervised
reconstruction head (trained on normal-riding windows only) and a supervised classification
head (trained on all labeled windows, using the reconstruction error and engineered
accel/gyro/jerk stats as additional inputs).

All the data loading, unit/gravity fixes, resampling, and session-level splitting logic lives
in `src/` — see `../README.md` for what was wrong with the original pipeline and how this fixes it, including the `data/darus/` source added to bring in real per-timestep-labeled crash physics data.

Run this with the `.venv` kernel (`python -m venv .venv && .venv\Scripts\pip install -r requirements.txt`).

In [ ]:
import json

import numpy as np
import tensorflow as tf

from src import config, io_loaders, preprocess, manifest, normalize, model as model_lib, evaluate

## Load and canonicalize all sessions

CARLA + road recordings (normal) and near-fall + fall recordings (crash-like), all projected onto a common 50Hz grid, deg/s gyro units, and gravity-removed acceleration.

In [ ]:
sessions = io_loaders.load_all_sessions()
print(f"Loaded {len(sessions)} sessions:")
for source in sorted(set(s.source for s in sessions)):
    n = sum(1 for s in sessions if s.source == source)
    print(f"  {source}: {n} sessions")

sessions = preprocess.canonicalize_all(sessions)

## Session-level train/val/test split

Splitting happens on whole recordings (not windows), stratified by source+label, so windows from the same file never leak across splits.

In [ ]:
split_of = manifest.assign_splits(sessions)

train_ds = manifest.build_dataset(sessions, split_of, "train")
val_ds = manifest.build_dataset(sessions, split_of, "val")
test_ds = manifest.build_dataset(sessions, split_of, "test")

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    n_normal = int((ds.y == 0).sum())
    n_anomaly = int((ds.y == 1).sum())
    print(f"  {name}: {len(ds.y)} windows ({n_normal} normal, {n_anomaly} anomaly)")

## Normalize

Mean/std computed from **normal-labeled training windows only** — the autoencoder's job is to model what normal looks like.

In [ ]:
normalizer = normalize.Normalizer.fit(train_ds.X_seq, train_ds.X_aux, train_ds.y)
config.MODELS_DIR.mkdir(exist_ok=True, parents=True)
normalizer.save(config.MODELS_DIR / "normalizer.npz")

X_seq_train, X_aux_train = normalizer.transform(train_ds.X_seq, train_ds.X_aux)
X_seq_val, X_aux_val = normalizer.transform(val_ds.X_seq, val_ds.X_aux)

recon_w_train, clf_w_train = model_lib.compute_sample_weights(train_ds.y)
recon_w_val, clf_w_val = model_lib.compute_sample_weights(val_ds.y)

## Build and train the model

In [ ]:
model = model_lib.build_model(recon_weight=2.0, clf_weight=1.0)
model.summary()

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_crash_prob_auc", mode="max", patience=8, restore_best_weights=True
)

history = model.fit(
    {"seq": X_seq_train, "aux": X_aux_train},
    {
        "reconstruction": X_seq_train,
        "crash_prob": train_ds.y,
        "recon_error": np.zeros((len(train_ds.y), 1), dtype="float32"),
    },
    sample_weight={
        "reconstruction": recon_w_train,
        "crash_prob": clf_w_train,
        "recon_error": np.ones(len(train_ds.y), dtype="float32"),
    },
    validation_data=(
        {"seq": X_seq_val, "aux": X_aux_val},
        {
            "reconstruction": X_seq_val,
            "crash_prob": val_ds.y,
            "recon_error": np.zeros((len(val_ds.y), 1), dtype="float32"),
        },
        {
            "reconstruction": recon_w_val,
            "crash_prob": clf_w_val,
            "recon_error": np.ones(len(val_ds.y), dtype="float32"),
        },
    ),
    epochs=60,
    batch_size=128,
    shuffle=True,
    callbacks=[early_stop],
    verbose=2,
)

In [ ]:
model.save(config.MODELS_DIR / "helmx_crash_model.keras")
with open(config.MODELS_DIR / "training_history.json", "w") as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f, indent=2)

## Calibrate decision thresholds on the validation split

Three operating points are saved: a balanced (F1-optimal) threshold, a moderate-recall threshold, and a high-recall threshold for sensitivity analysis. The deployment default is the balanced threshold loaded from `models/thresholds.json`. See `../results/metrics/test_metrics.json` after running `model-check.ipynb` for how these trade off on the held-out test set.

In [ ]:
val_preds = model.predict({"seq": X_seq_val, "aux": X_aux_val}, verbose=0)
val_crash_prob = val_preds["crash_prob"].ravel()
val_recon_error = val_preds["recon_error"].ravel()

normal_mask = val_ds.y == 0
recon_threshold = float(np.percentile(val_recon_error[normal_mask], 99))

thresholds = {
    "recon_error": recon_threshold,
    "classifier_prob_balanced": evaluate.choose_threshold_by_f1(val_ds.y, val_crash_prob),
    "classifier_prob_moderate_recall": evaluate.choose_threshold_by_recall(val_ds.y, val_crash_prob, target_recall=0.60),
    "classifier_prob_high_recall": evaluate.choose_threshold_by_recall(val_ds.y, val_crash_prob, target_recall=0.90),
}
thresholds["classifier_prob"] = thresholds["classifier_prob_balanced"]

with open(config.MODELS_DIR / "thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=2)
print("Thresholds:", thresholds)